In [61]:
from pathlib import Path
from scipy.optimize import linear_sum_assignment
from sklearn.metrics import average_precision_score

import pandas as pd
import numpy as np
import shutil
import csv

In [62]:
pd.set_option('display.expand_frame_repr', False)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

In [63]:
### Directories
project_root = Path.cwd()

summary_ground_truth_directory = project_root / Path("eval/gt_summary")
summary_predicted_directory = project_root / Path("eval/pred_summary")
frames_ground_truth_directory = project_root / Path("eval/gt_frames")
frames_predicted_directory = project_root / Path("eval/pred_frames")
result_directory = project_root / Path("eval/result")

summary_ground_truth_directory.mkdir(parents=True, exist_ok=True)
summary_predicted_directory.mkdir(parents=True, exist_ok=True)
result_directory.mkdir(parents=True, exist_ok=True)

print(f"Current Working Directory: {project_root}")

Current Working Directory: c:\Gabriel_Files\Programming_Files\School\Thesis\src


In [64]:
# for file in sorted(summary_ground_truth_directory.glob("*.csv")):
#     #print(file.stem)
#     file_stem = file.stem[7:-3]
#     if "tags" in file.name:
#         new_name = f"{file_stem}_summary_pred.csv"
#         shutil.copy(file, file.with_name(new_name))

# for file in sorted(summary_predicted_directory.glob("*.csv")):
#     print(file.stem)

# for file in sorted(summary_ground_truth_directory.glob("*.csv")):
#     file.rename(file.with_name(file.name.replace('pred', 'true')))

# cols = "frame_start,frame_end,human_id,object_id"
# for file in sorted(summary_ground_truth_directory.glob("*.csv")):
#     lines = file.read_text().splitlines(True)
#     if lines:
#         lines[0] = cols + "\n"
#         file.write_text("".join(lines))

vid_height = 1214


In [65]:
### Clear Existing Output Directories
for item in result_directory.iterdir():
    if item.is_file() or item.is_symlink():
        item.unlink()
    elif item.is_dir():
        shutil.rmtree(item)

In [66]:
def calculate_prec_rec_f1(results_df, total_preds, threshold=0.5):
    tp_rows = results_df[results_df['tiou'] >= threshold]
    tp = len(tp_rows)
    matched_preds = len(set(idx for indices in tp_rows['pred_indices'] for idx in indices))
    fp = total_preds - matched_preds
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / len(results_df) if len(results_df) > 0 else 0.0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    return precision, recall, f1

def evaluate_tiou(truth_csv_path, predicted_csv_path):
    import pandas as pd
    import numpy as np

    truth_df = pd.read_csv(truth_csv_path)
    pred_df = pd.read_csv(predicted_csv_path)
    matches = []

    for (hum_id, obj_id), truth_group in truth_df.groupby(['human_id', 'object_id']):
        pred_group = pred_df[(pred_df['human_id'] == hum_id) & (pred_df['object_id'] == obj_id)]
        
        if pred_group.empty:
            for t_idx, t_row in truth_group.iterrows():
                t_start = t_row['frame_start']
                t_end = t_row['frame_end']
                matches.append((hum_id, obj_id, t_idx, [], f"{t_start} - {t_end}", [], 0, (t_end - t_start) + 1, 0.0))
            continue
            
        for t_idx, t_row in truth_group.iterrows():
            t_start = t_row['frame_start']
            t_end = t_row['frame_end']
            
            p_starts = pred_group['frame_start'].values
            p_ends = pred_group['frame_end'].values
            
            overlap_starts = np.maximum(t_start, p_starts)
            overlap_ends = np.minimum(t_end, p_ends)
            overlap_durations = np.maximum(0, (overlap_ends - overlap_starts) + 1)
            
            valid_mask = overlap_durations > 0
            
            if not np.any(valid_mask):
                matches.append((hum_id, obj_id, t_idx, [], f"{t_start} - {t_end}", [], 0, (t_end - t_start) + 1, 0.0))
                continue
                
            v_p_starts = p_starts[valid_mask]
            v_p_ends = p_ends[valid_mask]
            v_overlap_durations = overlap_durations[valid_mask]
            
            intersection = np.sum(v_overlap_durations)
            
            t_duration = (t_end - t_start) + 1
            p_duration = np.sum((v_p_ends - v_p_starts) + 1)
            union = t_duration + p_duration - intersection
            
            tiou = intersection / union if union > 0 else 0
            
            t_frames = f"{t_start} - {t_end}"
            p_frames = [f"{s} - {e}" for s, e in zip(v_p_starts, v_p_ends)]
            p_indices = pred_group.index[valid_mask].tolist()
            
            matches.append((hum_id, obj_id, t_idx, p_indices, t_frames, p_frames, intersection, union, tiou))

    result_df = pd.DataFrame(matches, columns=['human_id', 'object_id', 'truth_idx', 'pred_indices', 'truth_frames', 'pred_frames', 'intersection', 'union', 'tiou'])
    return result_df

In [67]:
def evaluate_frame_level_score(truth_csv_path, predicted_csv_path):
    import pandas as pd
    t = set(map(tuple, pd.read_csv(truth_csv_path)[['frame_index', 'human_id', 'object_id']].values))
    p = set(map(tuple, pd.read_csv(predicted_csv_path)[['frame_index', 'human_id', 'object_id']].values))
    tp = len(t & p)
    precision = tp / len(p) if p else 0.0
    recall = tp / len(t) if t else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return precision, recall, f1

In [68]:
def convert_truth_to_frames(csv_path, video_stride):
    df = pd.read_csv(csv_path)
    frame_interactions = [["frame_index", "human_id", "object_id", "distance"]]
    for _, row in df.iterrows():
        start = int(row["frame_start"])
        end = int(row["frame_end"])
        aligned_start = start + (video_stride - start % video_stride) % video_stride
        for i in range(aligned_start, end, video_stride):
            frame_interactions.append([i, int(row["human_id"]), int(row["object_id"]), row.get("distance", "n/a")])
    return frame_interactions

In [69]:
# ### Evaluate specific vid

# video_name = "vid01"

# gt_filename_format = "_summary_true.csv"
# pred_filename_format = "_summary_log.csv"

# gt_csv_file = summary_ground_truth_directory / (video_name + gt_filename_format)
# pred_csv_file = summary_predicted_directory / (video_name + pred_filename_format)

# result_df = evaluate_tiou(gt_csv_file, pred_csv_file)

# total_preds = len(pd.read_csv(pred_csv_file))
# tp_rows = result_df[result_df['tiou'] >= 0.5]
# precision, recall, f1 = calculate_prec_rec_f1(result_df, total_preds, 0.5)

# logs = []
# logs.append(result_df.drop(columns=['truth_idx', 'pred_indices']).to_string())
# logs.append(f"Total Predictions: {total_preds}")
# logs.append(f"Mean TIOU: {result_df['tiou'].mean()}")
# logs.append(f"Precision: {precision}")
# logs.append(f"Recall: {recall}")
# logs.append(f"F1: {f1}")

# for log in logs:
#     print(log)

# with open(result_directory / f"{video_name}_result_summary", "w") as log_file:
#     log_file.writelines("\n".join(logs))

In [70]:
### Evaluate list of vids

video_skip_list = [13,20]
skip_names = {f"vid{s:02d}" for s in video_skip_list}

for csv in sorted(summary_ground_truth_directory.glob("*.csv")):
    video_name = csv.stem.split('_')[0]

    if video_name in skip_names:
        continue

    gt_filename_format = "_summary_true.csv"
    pred_filename_format = "_summary_log.csv"

    gt_csv_file = summary_ground_truth_directory / (video_name + gt_filename_format)
    pred_csv_file = summary_predicted_directory / (video_name + pred_filename_format)

    result_df = evaluate_tiou(gt_csv_file, pred_csv_file)

    total_preds = len(pd.read_csv(pred_csv_file))
    tp_rows = result_df[result_df['tiou'] >= 0.5]
    precision, recall, f1 = calculate_prec_rec_f1(result_df, total_preds, 0.5)

    logs = []
    logs.append(result_df.drop(columns=['truth_idx', 'pred_indices']).to_string())
    logs.append(f"Total Predictions: {total_preds}")
    logs.append(f"Mean TIOU: {result_df['tiou'].mean()}")
    logs.append(f"Precision: {precision}")
    logs.append(f"Recall: {recall}")
    logs.append(f"F1: {f1}")

    for log in logs:
        print(log)

    with open(result_directory / f"{video_name}_result_summary", "w") as log_file:
        log_file.writelines("\n".join(logs))


   human_id  object_id truth_frames    pred_frames  intersection  union      tiou
0         1          4      0 - 105      [0 - 100]           101    106  0.952830
1         1          4   280 - 3921   [280 - 3920]          3641   3642  0.999725
2         3          6   140 - 1340   [150 - 1335]          1186   1201  0.987510
3         3         12  2445 - 2905  [2445 - 2900]           456    461  0.989154
Total Predictions: 8
Mean TIOU: 0.982305008819526
Precision: 0.5
Recall: 1.0
F1: 0.6666666666666666
   human_id  object_id truth_frames  pred_frames  intersection  union      tiou
0         1          1      0 - 220    [0 - 215]           216    221  0.977376
1         1          1    370 - 445  [370 - 440]            71     76  0.934211
2         2          4    420 - 460  [420 - 455]            36     41  0.878049
Total Predictions: 3
Mean TIOU: 0.929878290804818
Precision: 1.0
Recall: 1.0
F1: 1.0
   human_id  object_id truth_frames pred_frames  intersection  union      tiou
0     

In [71]:
# video_name = "vid01"
# video_stride = 5

# gt_filename_format = "_summary_true.csv"
# gt_frames_filename_format = "_interaction_true.csv"

# gt_csv_file = summary_ground_truth_directory / (video_name + gt_filename_format)

# frames = convert_truth_to_frames(gt_csv_file, video_stride)
# frames_flattened = [f"{f_id},{h_id},{o_id},{d}" for (f_id, h_id, o_id, d) in frames]
# with open(frames_ground_truth_directory / f"{video_name}{gt_frames_filename_format}", "w") as frames_truth_csv:
#     frames_truth_csv.writelines("\n".join(frames_flattened))

In [72]:
video_stride = 5

for csv in sorted(summary_ground_truth_directory.glob("*.csv")):
    video_name = csv.stem.split('_')[0]
    
    if video_name in skip_names:
        continue

    gt_filename_format = "_summary_true.csv"
    gt_frames_filename_format = "_interaction_true.csv"

    gt_csv_file = summary_ground_truth_directory / (video_name + gt_filename_format)

    frames = convert_truth_to_frames(gt_csv_file, video_stride)
    frames_flattened = [f"{f_id},{h_id},{o_id},{d}" for (f_id, h_id, o_id, d) in frames]
    with open(frames_ground_truth_directory / f"{video_name}{gt_frames_filename_format}", "w") as frames_truth_csv:
        frames_truth_csv.writelines("\n".join(frames_flattened))

In [73]:
for csv in sorted(summary_ground_truth_directory.glob("*.csv")):
    video_name = csv.stem.split('_')[0]

    if video_name in skip_names:
        continue

    gt_filename_format = "_interaction_true.csv"
    pred_filename_format = "_interaction_log.csv"

    gt_csv_file = frames_ground_truth_directory / (video_name + gt_filename_format)
    pred_csv_file = frames_predicted_directory / (video_name + pred_filename_format)

    precision, recall, f1 = evaluate_frame_level_score(gt_csv_file, pred_csv_file)

    logs = []
    logs.append(f"Precision: {precision}")
    logs.append(f"Recall: {recall}")
    logs.append(f"F1: {f1}")

    for log in logs:
        print(log)

    with open(result_directory / f"{video_name}_result_frames", "w") as log_file:
        log_file.writelines("\n".join(logs))

Precision: 0.911738746690203
Recall: 0.9547134935304991
F1: 0.9327313769751692
Precision: 0.9027777777777778
Recall: 0.9701492537313433
F1: 0.9352517985611511
Precision: 0.9930232558139535
Recall: 0.9748858447488584
F1: 0.9838709677419354
Precision: 0.9825289635090763
Recall: 0.9499251272836179
F1: 0.9659520038981605
Precision: 0.5656565656565656
Recall: 0.415327564894932
F1: 0.47897362794012827
Precision: 0.0
Recall: 0.0
F1: 0.0
Precision: 0.876979293544458
Recall: 0.9195402298850575
F1: 0.8977556109725685
Precision: 0.8178633975481612
Recall: 0.6700143472022956
F1: 0.7365930599369086
Precision: 0.8532035685320357
Recall: 0.9669117647058824
F1: 0.9065058164584232
Precision: 0.9134709931170109
Recall: 0.9360201511335012
F1: 0.9246081114705151
Precision: 0.8111111111111111
Recall: 0.6239316239316239
F1: 0.7053140096618358
Precision: 1.0
Recall: 0.9405010438413361
F1: 0.9693383539537386
Precision: 0.9849884526558892
Recall: 0.8931937172774869
F1: 0.9368478857770456
Precision: 0.986151368